# Discussion 03 Notebook

This notebook is an accompaniment to the associated discussion worksheet handout.

## Load in the IMDB Performance database

This is a variation of the IMDB database with keys defined. Note that this is a pretty big database! So if you run the below lines, please also remember to delete the `imdb_perf_lecture` afterwards to save space on your limited postgreSQL server.

We assume you have the associated lecture folder `lec06` pulled into your repo already. The below commands create a symbolic link (i.e., shortcut/redirect with `ln`) to this lecture data directory, allowing some space saving, and unzip the database file.

In [5]:
!psql -h localhost -c 'DROP DATABASE IF EXISTS imdb_perf_lecture'
!psql -h localhost -c 'CREATE DATABASE imdb_perf_lecture' 
!psql -h localhost -d imdb_perf_lecture -f imdb_perf_lecture.sql

ERROR:  database "imdb_perf_lecture" is being accessed by other users
DETAIL:  There are 3 other sessions using the database.
ERROR:  database "imdb_perf_lecture" already exists
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
psql:imdb_perf_lecture.sql:30: ERROR:  relation "actors" already exists
ALTER TABLE
psql:imdb_perf_lecture.sql:42: ERROR:  relation "cast_info" already exists
ALTER TABLE
psql:imdb_perf_lecture.sql:56: ERROR:  relation "movies" already exists
ALTER TABLE
psql:imdb_perf_lecture.sql:845954: ERROR:  duplicate key value violates unique constraint "actor_pkey"
DETAIL:  Key (id)=(1) already exists.
CONTEXT:  COPY actors, line 1
COPY 2211936
psql:imdb_perf_lecture.sql:3714359: ERROR:  duplicate key value violates unique constraint "movie_pkey"
DETAIL:  Key (id)=(9) already exists.
CONTEXT:  COPY movies, line 1
psql:imdb_perf_lecture.sql:3714367: ERROR:  multiple primary keys for table "actors" are not allowed
psql:imdb_perf_lecture.sql:37

Before starting this part, review the schema of the relations in the `imdb_perf_lecture` database. Here's the printout from `psql`:

```
imdb_perf_lecture=# \d actors
               Table "public.actors"
 Column |  Type   | Collation | Nullable | Default 
--------+---------+-----------+----------+---------
 id     | integer |           | not null | 
 name   | text    |           |          | 
Indexes:
    "actor_pkey" PRIMARY KEY, btree (id)
Referenced by:
    TABLE "cast_info" CONSTRAINT "cast_info_person_id_fkey" FOREIGN KEY (person_id) REFERENCES actors(id)

imdb_perf_lecture=# \d movies
                   Table "public.movies"
     Column      |  Type   | Collation | Nullable | Default 
-----------------+---------+-----------+----------+---------
 id              | integer |           | not null | 
 title           | text    |           |          | 
 year            | integer |           |          | 
 runtime_minutes | integer |           |          | 
Indexes:
    "movie_pkey" PRIMARY KEY, btree (id)
Referenced by:
    TABLE "cast_info" CONSTRAINT "cast_info_movie_id_fkey" FOREIGN KEY (movie_id) REFERENCES movies(id)

imdb_perf_lecture=# \d cast_info
               Table "public.cast_info"
  Column   |  Type   | Collation | Nullable | Default 
-----------+---------+-----------+----------+---------
 person_id | integer |           |          | 
 movie_id  | integer |           |          | 
Foreign-key constraints:
    "cast_info_movie_id_fkey" FOREIGN KEY (movie_id) REFERENCES movies(id)
    "cast_info_person_id_fkey" FOREIGN KEY (person_id) REFERENCES actors(id)

```

In [6]:
%reload_ext sql
%sql postgresql://127.0.0.1:5432/imdb_perf_lecture
import pandas as pd


# IV. Query Performance

This question looks at the impacts of **aggregation** and **sorting** on query performance.

In [8]:
%%sql
SELECT * FROM actors;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

845888 rows affected.

id,name
1,Fred Astaire
2,Lauren Bacall
3,Brigitte Bardot
4,John Belushi
5,Ingmar Bergman
6,Ingrid Bergman
7,Humphrey Bogart
8,Marlon Brando
9,Richard Burton
10,James Cagney


## Question 8

Write a query that returns the actor names and the number of times the corresponding name appears in the `actors` relation.

In [7]:
%%sql
-- write your query here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

845888 rows affected.

id,name
1,Fred Astaire
2,Lauren Bacall
3,Brigitte Bardot
4,John Belushi
5,Ingmar Bergman
6,Ingrid Bergman
7,Humphrey Bogart
8,Marlon Brando
9,Richard Burton
10,James Cagney


## Question 9

Write a query that returns the actor IDs and the number of times the corresponding ID appears in the `actors` relation.

In [6]:
%%sql
-- write your query here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

845888 rows affected.

id,count
1,1
2,1
3,1
4,1
5,1
6,1
7,1
8,1
9,1
10,1


## Question 10

Run `EXPLAIN ANALYZE` on your two queries above. See below for the full question.

If you're having trouble seeing the entirety of the query plan, you can run the following cell to set the limit on displayed rows to 20. **Careful**: Do not set this to `None` and run the actual queries; SQL will return millions of rows and crash your kernel!

In [6]:
# run this cell to remove 10-row limit on display
%config SqlMagic.displaylimit = 20


In [7]:
%%sql
-- write your EXPLAIN ANALYZE here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

6 rows affected.

QUERY PLAN
HashAggregate (cost=67874.58..83327.62 rows=719241 width=22) (actual time=309.557..598.633 rows=804435 loops=1)
Group Key: name
Planned Partitions: 32 Batches: 33 Memory Usage: 4113kB Disk Usage: 30624kB
-> Seq Scan on actors (cost=0.00..13684.88 rows=845888 width=14) (actual time=0.041..58.525 rows=845888 loops=1)
Planning Time: 0.069 ms
Execution Time: 629.172 ms


In [8]:
%%sql
-- write your EXPLAIN ANALYZE here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

6 rows affected.

QUERY PLAN
GroupAggregate (cost=0.42..34669.07 rows=845888 width=12) (actual time=0.026..247.602 rows=845888 loops=1)
Group Key: id
-> Index Only Scan using actor_pkey on actors (cost=0.42..21980.74 rows=845888 width=4) (actual time=0.021..75.903 rows=845888 loops=1)
Heap Fetches: 0
Planning Time: 0.075 ms
Execution Time: 271.234 ms


<br/><br/>

**(Question, continued)**
Why do you think the the `name` query use a Sequential Scan, whereas the `id` query use an Index Only scan?

## Question 11

Write a command that creates an index `name_actor_index` on the `name` attribute of `actors`.

In [6]:
%%sql
-- write your query here --
-- CREATE INDEX name_actor_index on actors (name);

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

++
||
++
++

In [14]:
# DROP INDEX name_actor_index 

**(Question, continued)**
Rerun your `EXPLAIN ANALYZE` of your Question 3 query on `name` by copying and pasting it into the cell below. See below for the discussion question.

In [10]:
%%sql
-- write your EXPLAIN ANALYZE here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

6 rows affected.

QUERY PLAN
GroupAggregate (cost=0.42..37614.60 rows=719241 width=22) (actual time=0.064..319.919 rows=804435 loops=1)
Group Key: name
-> Index Only Scan using name_actor_index on actors (cost=0.42..26192.75 rows=845888 width=14) (actual time=0.055..98.464 rows=845888 loops=1)
Heap Fetches: 0
Planning Time: 0.187 ms
Execution Time: 343.520 ms


**(Question, continued)**
Why does the `name` query now use an Index Only Scan? What index is it using?

## Question 12

Analyze the impact of sorting as follows. Rewrite your query from **Question 4** to return the entries sorted by ID. In other words, run an `EXPLAIN ANALYZE` on a query that returns the actor IDs (**sorted by lowest ID first**) and the number of times the corresponding ID appears in the `actors` relation.

**Discuss**: Do you expect this query to take more time? Why or why not?

In [12]:
%%sql
-- write your query here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

RuntimeError: (psycopg2.ProgrammingError) can't execute an empty query
[SQL: -- write your query here --]
(Background on this error at: https://sqlalche.me/e/20/f405)
If you need help solving this issue, send us a message: https://ploomber.io/community


# V. Table Sampling

If you're having trouble seeing the entirety of the query plan, you can run the following cell to set the limit on displayed rows to 20. **Careful**: Do not set this to `None` and run the actual queries; SQL will return millions of rows and crash your kernel!

Consider the following query which randomly selects 10,000 rows from the `actors` table:
`SELECT *
FROM actors
ORDER BY RANDOM()
LIMIT 10000;`

In [15]:
# run this cell to change default 10-row limit on display
%config SqlMagic.displaylimit = 20

#### ORDER BY sampling

In [5]:
%%sql
EXPLAIN ANALYZE
SELECT *
FROM actors
ORDER BY RANDOM()
LIMIT 10000;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

7 rows affected.

QUERY PLAN
Limit (cost=76228.62..76253.62 rows=10000 width=44) (actual time=231.591..233.233 rows=10000 loops=1)
-> Sort (cost=76228.62..78343.34 rows=845888 width=44) (actual time=231.590..232.682 rows=10000 loops=1)
Sort Key: (random())
Sort Method: top-N heapsort Memory: 2092kB
-> Seq Scan on actors (cost=0.00..15799.60 rows=845888 width=44) (actual time=0.050..97.637 rows=845888 loops=1)
Planning Time: 0.283 ms
Execution Time: 234.347 ms


## Question 13

Write a query that uses `TABLESAMPLE BERNOULLI` to randomly select 10,000 rows from `actors`, on average. Hint: Use a subquery to compute the total rows in `actors`.

In [12]:
%%sql
-- write your query here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

8432 rows affected.

id,name
34,William Holden
63,Anthony Quinn
84,Gong Li
187,Madonna
448,Lance Henriksen
681,Vince Vaughn
759,Paul Thomas Anderson
931,Elizabeth Berridge
1118,Pam Dawber
1236,Meg Foster


## Question 14

Write a query that uses `TABLESAMPLE SYSTEM` to randomly select 10,000 rows from `actors`, on average.
How does this page-level table sampling compare with the Bernoulli table sampling in the previous
question?

In [19]:
%%sql
-- write your query here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

7257 rows affected.

id,name
5212,Ian McKellen
5214,Sarah McLachlan
5215,Jenny McShane
5216,Janet McTeer
5217,Jayne Meadows
5218,Tim Meadows
5219,Mike Medavoy
5220,Tamara Mello
5221,Christopher Meloni
5222,Sam Mendes


## Question 15

Run `EXPLAIN ANALYZE` on each of the three sampling methods above. Does this confirm your
understanding of the different methods?

In [20]:
%%sql


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

7 rows affected.

QUERY PLAN
Limit (cost=76228.62..76253.62 rows=10000 width=26) (actual time=220.177..221.763 rows=10000 loops=1)
-> Sort (cost=76228.62..78343.34 rows=845888 width=26) (actual time=220.176..221.216 rows=10000 loops=1)
Sort Key: (random())
Sort Method: top-N heapsort Memory: 2094kB
-> Seq Scan on actors (cost=0.00..15799.60 rows=845888 width=26) (actual time=0.045..90.474 rows=845888 loops=1)
Planning Time: 0.076 ms
Execution Time: 222.679 ms


In [6]:
%%sql
-- write your EXPLAIN ANALYZE here --


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

11 rows affected.

QUERY PLAN
Sample Scan on actors (cost=10631.89..16703.78 rows=84589 width=18) (actual time=32.848..77.197 rows=8434 loops=1)
Sampling: bernoulli ((1000000 / $1))
InitPlan 1 (returns $1)
-> Finalize Aggregate (cost=10631.88..10631.89 rows=1 width=8) (actual time=32.822..32.925 rows=1 loops=1)
-> Gather (cost=10631.67..10631.88 rows=2 width=8) (actual time=32.750..32.918 rows=3 loops=1)
Workers Planned: 2
Workers Launched: 2
-> Partial Aggregate (cost=9631.67..9631.68 rows=1 width=8) (actual time=30.418..30.419 rows=1 loops=3)
-> Parallel Seq Scan on actors actors_1 (cost=0.00..8750.53 rows=352453 width=0) (actual time=0.021..18.489 rows=281963 loops=3)
Planning Time: 0.232 ms


In [22]:
%%sql


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

11 rows affected.

QUERY PLAN
Sample Scan on actors (cost=10631.89..13569.78 rows=84589 width=18) (actual time=33.540..35.512 rows=8903 loops=1)
Sampling: system ((1000000 / $1))
InitPlan 1 (returns $1)
-> Finalize Aggregate (cost=10631.88..10631.89 rows=1 width=8) (actual time=33.503..34.610 rows=1 loops=1)
-> Gather (cost=10631.67..10631.88 rows=2 width=8) (actual time=33.409..34.604 rows=3 loops=1)
Workers Planned: 2
Workers Launched: 2
-> Partial Aggregate (cost=9631.67..9631.68 rows=1 width=8) (actual time=30.921..30.922 rows=1 loops=3)
-> Parallel Seq Scan on actors actors_1 (cost=0.00..8750.53 rows=352453 width=0) (actual time=0.033..19.011 rows=281963 loops=3)
Planning Time: 0.103 ms


## Question 16 [Optional]:

In the above queries, computing the total rows in `actors` for every table sample takes time. Use
`EXPLAIN ANALYZE` to compare with using the PostgreSQL estimates of numbers of records: <br>
`SELECT reltuples FROM pg_class where relname = ‘actors’;` <br>
Rewrite your table sampling queries above to use this row estimate, and note the speedup in performance.

In [23]:
%%sql
-- write your query here --
EXPLAIN ANALYZE SELECT reltuples FROM pg_class where relname = 'actors';

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Index Scan using pg_class_relname_nsp_index on pg_class (cost=0.27..8.29 rows=1 width=4) (actual time=0.012..0.013 rows=1 loops=1)
Index Cond: (relname = 'actors'::name)
Planning Time: 16.166 ms
Execution Time: 0.038 ms
